In [2]:
from data_loader import get_prices, get_returns, SPLIT
prices = get_prices(start=SPLIT)
prices.head()

,Gold,Silver,Oil,S&P 500
Date,,,,
2025-01-02,2669.0000,29.622,75.93,5868.5498
2025-01-03,2654.7000,29.806,76.51,5942.4702
2025-01-06,2647.3999,30.342,76.30,5975.3799
2025-01-07,2665.3999,30.447,77.05,5909.0298
2025-01-08,2672.3999,30.452,76.16,5918.2500


In [3]:
returns = get_returns(start=SPLIT)
returns.head()

,Gold,Silver,Oil,S&P 500
Date,,,,
2025-01-03,-0.005372,0.006192,0.007610,0.012517
2025-01-06,-0.002754,0.017823,-0.002749,0.005523
2025-01-07,0.006776,0.003455,0.009782,-0.011166
2025-01-08,0.002623,0.000164,-0.011618,0.001559
2025-01-10,0.015815,0.020767,0.046186,-0.015532


In [4]:
x = returns["Gold"]
y = returns["Silver"]
print("Number of days:", len(x))

Number of days: 458


In [5]:
import numpy as np

x_bar = np.mean(x)
y_bar = np.mean(y)


print("Average gold return:  ", x_bar)
print("Average silver return:", y_bar)

Average gold return:   0.0008274903187750191
Average silver return: 0.0011032327612067498


In [6]:
regressors = ["Gold"]                                       # might add Oil, S&P 500 or other x variables here
n = len(y)                                                  # number of days

Y = y.to_numpy()                                            # silver returns as a vector
X = np.column_stack([np.ones(n), returns[regressors]])      # [1, regressors] matrix

k = len(regressors)                                         # number of regressors
df = n - k - 1                                              # degrees of freedom

print("n:", n, "| k:", k, "| df:", df)
print("Y shape:", Y.shape)
print("X shape:", X.shape)

n: 458 | k: 1 | df: 456
Y shape: (458,)
X shape: (458, 2)


In [7]:
import pandas as pd

X_prime_X = X.T @ X                              # X′X
X_prime_Y = X.T @ Y                              # X′Y
X_prime_X_inv = np.linalg.inv(X_prime_X)         # (X′X)⁻¹

beta_hat = X_prime_X_inv @ X_prime_Y             # β̂ = (X′X)⁻¹X′Y

beta_labels = ["Intercept"] + regressors
coefficients = pd.Series(beta_hat, index=beta_labels)
print(coefficients.round(4))

Intercept   -0.0003
Gold         1.6800
dtype: float64


In [8]:
Y_hat = X @ beta_hat                          # Ŷ = Xβ̂  (fitted values)
u_hat = Y - Y_hat                             # û = Y − Ŷ  (residuals)

SSR = u_hat @ u_hat                           # SSR = Σû²  (sum of squared residuals)
sigma_sq_hat = SSR / df                       # σ̂² = SSR / (n − k − 1)  (estimated variance of errors)
sigma_hat = np.sqrt(sigma_sq_hat)             # σ̂  (estimated standard deviation of the errors)

var_beta_hat = sigma_sq_hat * X_prime_X_inv   # var(β̂) = σ̂²(X′X)⁻¹
se_beta_hat = np.sqrt(np.diag(var_beta_hat))  # se(β̂₁) = σ̂ / √Σ(xᵢ − x̄)²  (i use matrix version: sqrt of diagonal)

results = pd.DataFrame({
    "Coefficient": beta_hat,
    "Std. Error": se_beta_hat
}, index=beta_labels)

print("SSR:      ", SSR)
print("sigma_hat:", sigma_hat)
print()
print(results.round(4))

# Check: se(β̂₁) alternative formula, should match the Gold row above
SST_x = np.sum((x - x_bar) ** 2)              # Σ(xᵢ − x̄)²
print()
print("se(β̂₁) alternative formula:", sigma_hat / np.sqrt(SST_x))

# Sanity checks: both should be near 0 (OLS properties with an intercept)
print()
print("Mean of residuals:", np.mean(u_hat))
print("X′û (near 0):", X.T @ u_hat)

SSR:       0.22293790932283547
sigma_hat: 0.022111058860976935

           Coefficient  Std. Error
Intercept      -0.0003       0.001
Gold            1.6800       0.065

se(β̂₁) alternative formula: 0.06497226207146911

Mean of residuals: -6.060169348390592e-19
X′û (near 0): [-4.99600361e-16 -4.85722573e-17]


In [9]:
from scipy import stats

alpha_level = 0.05                                   # testing at the 5% level
t_crit = stats.t.ppf(1 - alpha_level / 2, df)        # critical value for a two-sided test 

# Test 1: is silver linked to gold at all (H0: beta = 0)
t_beta_0 = beta_hat / se_beta_hat                    # how many SEs beta_hat is away from 0
p_beta_0 = 2 * stats.t.sf(np.abs(t_beta_0), df)      # p-value, both tails

# Test 2: does silver move one-for-one with gold (H0: beta = 1)
t_beta_1 = (beta_hat - 1) / se_beta_hat              # how many SEs beta_hat is away from 1
p_beta_1 = 2 * stats.t.sf(np.abs(t_beta_1), df)      # p-value, both tails

results["t (β=0)"] = t_beta_0
results["p (β=0)"] = p_beta_0
results["Reject β=0"] = (np.abs(t_beta_0) > t_crit).astype(object)   # reject if |t| is bigger than the critical value

results["t (β=1)"] = t_beta_1
results["p (β=1)"] = p_beta_1
results["Reject β=1"] = (np.abs(t_beta_1) > t_crit).astype(object)

# Test 2 doesn't make sense for the intercept, so blanking it out
results.loc["Intercept", ["t (β=1)", "p (β=1)", "Reject β=1"]] = np.nan

print(f"Critical value (5%, df = {df}): {t_crit:.3f}")
print()

for name in regressors:
    row = results.loc[name]
    verdict_0 = "reject" if row["Reject β=0"] else "fail to reject"
    verdict_1 = "reject" if row["Reject β=1"] else "fail to reject"

    print(f"{name}: beta = {row['Coefficient']:.2f}  (se = {row['Std. Error']:.3f})")
    print(f"  H0: beta = 0  ->  t = {row['t (β=0)']:.2f},  p = {row['p (β=0)']:.4f}  ->  {verdict_0}")
    print(f"  H0: beta = 1  ->  t = {row['t (β=1)']:.2f},  p = {row['p (β=1)']:.4f}  ->  {verdict_1}")

results.round(4)

Critical value (5%, df = 456): 1.965

Gold: beta = 1.68  (se = 0.065)
  H0: beta = 0  ->  t = 25.86,  p = 0.0000  ->  reject
  H0: beta = 1  ->  t = 10.47,  p = 0.0000  ->  reject


,Coefficient,Std. Error,t (β=0),p (β=0),Reject β=0,t (β=1),p (β=1),Reject β=1
Intercept,-0.0003,0.001,-0.2774,0.7816,False,NaN,NaN,NaN
Gold,1.6800,0.065,25.8574,0.0000,True,10.4663,0.0,True


In [10]:
# 95% confidence interval: beta_hat +- t_crit x se
results["CI low"] = beta_hat - t_crit * se_beta_hat      # lower bound
results["CI high"] = beta_hat + t_crit * se_beta_hat     # upper bound

# R squared: how much of silver's movement the regression explains
SST = np.sum((Y - np.mean(Y)) ** 2)                      # total sum of squares, how much silver moves around its average
r_squared = 1 - SSR / SST                                # R² = 1 − SSR / SST

print(f"R²: {r_squared:.3f}")
print()

for name in regressors:
    row = results.loc[name]
    print(f"{name}: 95% CI for beta = [{row['CI low']:.2f}, {row['CI high']:.2f}]")

    if row["CI low"] > 1:
        print("  whole interval is above 1 -> silver moves more than one-for-one with gold")
    elif row["CI high"] < 1:
        print("  whole interval is below 1 -> silver moves less than gold")
    else:
        print("  interval includes 1 -> can't rule out a one-for-one move")

results.round(4)

R²: 0.595

Gold: 95% CI for beta = [1.55, 1.81]
  whole interval is above 1 -> silver moves more than one-for-one with gold


,Coefficient,Std. Error,t (β=0),p (β=0),Reject β=0,t (β=1),p (β=1),Reject β=1,CI low,CI high
Intercept,-0.0003,0.001,-0.2774,0.7816,False,NaN,NaN,NaN,-0.0023,0.0017
Gold,1.6800,0.065,25.8574,0.0000,True,10.4663,0.0,True,1.5523,1.8077


In [11]:
# Check my OLS against statsmodels
import statsmodels.api as sm

model = sm.OLS(Y, X).fit()
print(model.summary(xname=beta_labels))

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.595
Model:                            OLS   Adj. R-squared:                  0.594
Method:                 Least Squares   F-statistic:                     668.6
Date:                Tue, 24 Nov 2026   Prob (F-statistic):           2.00e-91
Time:                        15:33:28   Log-Likelihood:                 1096.9
No. Observations:                 458   AIC:                            -2190.
Df Residuals:                     456   BIC:                            -2181.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -0.0003      0.001     -0.277      0.7

In [12]:
all_returns = get_returns()


def run_ols(y, x, alpha_level=0.05):
    """OLS by matrix formula (same steps as Cells 5 to 9). Returns beta_hat, se, t_crit, R squared and n."""
    n = len(y)
    Y = y.to_numpy()
    X = np.column_stack([np.ones(n), x])              # [1, regressors]
    k = x.shape[1]                                    # number of regressors (columns of x), same as len(regressors)
    df = n - k - 1

    X_prime_X_inv = np.linalg.inv(X.T @ X)            # (X′X)⁻¹
    beta_hat = X_prime_X_inv @ (X.T @ Y)              # β̂ = (X′X)⁻¹X′Y

    u_hat = Y - X @ beta_hat                          # residuals
    SSR = u_hat @ u_hat
    sigma_sq_hat = SSR / df                           # estimated error variance
    se_beta_hat = np.sqrt(np.diag(sigma_sq_hat * X_prime_X_inv))

    t_crit = stats.t.ppf(1 - alpha_level / 2, df)
    SST = np.sum((Y - np.mean(Y)) ** 2)               # total sum of squares, how much silver moves around its average
    r_squared = 1 - SSR / SST                            # R² = 1 − SSR / SST
    return beta_hat, se_beta_hat, t_crit, r_squared, n


# adding more periods to compare
periods = {
    "2015-2024": ("2015-01-01", "2024-12-31"),
    "2025":      ("2025-01-01", "2025-12-31"),
    "2026":      ("2026-01-01", None),
}

rows = []
for label, (start, end) in periods.items():
    data = all_returns.loc[start:end]
    beta_hat_period, se_period, t_crit_period, r2_period, n_period = run_ols(data["Silver"], data[regressors])

    gold_beta, gold_se = beta_hat_period[1], se_period[1]     # [1] = gold row (0 is the intercept)
    rows.append({
        "Period": label,
        "Beta (gold)": gold_beta,
        "Std. Error": gold_se,
        "CI low": gold_beta - t_crit_period * gold_se,
        "CI high": gold_beta + t_crit_period * gold_se,
        "t (β=1)": (gold_beta - 1) / gold_se,
        "R²": r2_period,
        "n": n_period,
    })

by_period = pd.DataFrame(rows).set_index("Period")
by_period.round(3)

,Beta (gold),Std. Error,CI low,CI high,t (β=1),R²,n
Period,,,,,,,
2015-2024,1.506,0.024,1.459,1.554,20.753,0.603,2512
2025,1.166,0.087,0.995,1.337,1.911,0.421,250
2026,1.990,0.091,1.810,2.169,10.864,0.697,209


In [16]:
# Check my White robust (HC1) standard errors against statsmodels
X_arr = np.asarray(X)
Y_arr = np.asarray(Y)
n, p = X_arr.shape
X_prime_X_inv = np.linalg.inv(X_arr.T @ X_arr)
beta_hat = X_prime_X_inv @ (X_arr.T @ Y_arr)
u_hat = Y_arr - X_arr @ beta_hat
meat = (X_arr.T * u_hat ** 2) @ X_arr
my_se = np.sqrt(np.diag(X_prime_X_inv @ meat @ X_prime_X_inv * n / (n - p)))        # n - p is the same as n - k - 1

robust_model = sm.OLS(Y, X).fit(cov_type = "HC1")
print("statsmodels HC1 SE:", np.round(robust_model.bse, 4))
print("my HC1 SE:         ", np.round(my_se, 4))

statsmodels HC1 SE: [0.0011 0.1972]
my HC1 SE:          [0.0011 0.1972]
